# AI-Based Zodiac Wardrobe Picker — Final Colab Hosting Notebook

This notebook hosts your **already-trained MobileNetV2 model** with the Zodiac Wardrobe Picker website using **FastAPI + Uvicorn + pyngrok**.

## Required files

1. `zodiac_wardrobe_colab_ready_app.zip` — the companion application package.
2. `clothing_model_bundle.zip` — your trained `.keras` model and `labels.json` from Google Colab training.

## Optional file

3. Your Kaggle dataset ZIP. The notebook can extract it inside Colab if uploaded, but it is **not required** to host the model or make predictions.

**All ZIP extraction happens inside Google Colab. You do not need to extract the archives on your computer.**

Keep your ngrok authtoken ready. Never share it. This is a temporary public demo: the site stops when the Colab runtime or ngrok tunnel stops. Anyone with the ngrok URL can access it, so use only non-sensitive test images.

## 1. Install server and model dependencies

This cell is compatible with the current Colab Python runtime. It installs the web-server packages first, then **uses TensorFlow already present in Colab when available**. If TensorFlow is missing, it chooses a TensorFlow version compatible with the active Python version (including Python 3.13).

In [1]:
import sys, subprocess

# Install the web application dependencies. Do not pin TensorFlow here:
# Colab may already provide a compatible TensorFlow build.
server_packages = [
    "fastapi>=0.110,<1.0",
    "uvicorn[standard]>=0.27,<1.0",
    "python-multipart>=0.0.9",
    "pillow>=10.0",
    "pyngrok>=7.0",
    "requests>=2.31",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *server_packages])

# Reuse Colab's TensorFlow if it imports successfully.
try:
    import tensorflow as tf
    print("Using TensorFlow already available in this runtime:", tf.__version__)
except Exception as import_error:
    print("TensorFlow is not currently importable:", repr(import_error))
    # TensorFlow 2.21 provides Python 3.13 wheels. Older Python versions
    # can resolve a compatible release from the broader lower bound.
    if sys.version_info >= (3, 13):
        tensorflow_requirement = "tensorflow>=2.21,<2.22"
    elif sys.version_info >= (3, 12):
        tensorflow_requirement = "tensorflow>=2.20,<2.22"
    else:
        tensorflow_requirement = "tensorflow>=2.16"
    print("Installing", tensorflow_requirement)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", tensorflow_requirement])
    import tensorflow as tf

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices("GPU"))

Using TensorFlow already available in this runtime: 2.21.0
Python: 3.13.18 (main, Oct 26 2026, 15:28:15) [GCC 13.3.0]
TensorFlow: 2.21.0
Available GPUs: []


## 2. Upload the application ZIP and trained model ZIP

In the upload picker, select both required files:

- `zodiac_wardrobe_colab_ready_app.zip`
- `clothing_model_bundle.zip`

You may also select the Kaggle dataset ZIP as an optional third file. It will be extracted in Colab if present. For the fastest hosting setup, upload only the two required ZIPs.

In [2]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile, BadZipFile
import shutil

UPLOAD_DIR = Path('/content/zodiac_uploads')
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No files uploaded. Select the application ZIP and trained model ZIP.')

uploaded_paths = []
for name, content in uploaded.items():
    path = UPLOAD_DIR / Path(name).name
    path.write_bytes(content)
    uploaded_paths.append(path)

APP_ZIP = MODEL_ZIP = DATASET_ZIP = None
unknown = []
for path in uploaded_paths:
    try:
        with ZipFile(path) as zf:
            names = [n.replace('\\', '/') for n in zf.namelist()]
            has_backend = any(n.rstrip('/').endswith('backend/main.py') for n in names)
            has_index = any(Path(n).name.lower() == 'index.html' for n in names)
            has_model = any(n.lower().endswith('.keras') for n in names)
            has_labels = any(Path(n).name.lower() == 'labels.json' for n in names)
            has_styles = any(Path(n).name.lower() == 'styles.csv' for n in names)
            if has_backend and has_index:
                if APP_ZIP is not None: raise ValueError('More than one app ZIP found.')
                APP_ZIP = path
            elif has_model and has_labels:
                if MODEL_ZIP is not None: raise ValueError('More than one model ZIP found.')
                MODEL_ZIP = path
            elif has_styles:
                DATASET_ZIP = path
            else:
                unknown.append(path.name)
    except BadZipFile:
        unknown.append(path.name + ' (not a valid ZIP)')

if APP_ZIP is None:
    raise FileNotFoundError('App ZIP not found: expected backend/main.py and index.html.')
if MODEL_ZIP is None:
    raise FileNotFoundError('Model ZIP not found: expected one .keras file and labels.json.')
print('Application ZIP:', APP_ZIP.name)
print('Model bundle ZIP:', MODEL_ZIP.name)
print('Kaggle dataset ZIP:', DATASET_ZIP.name if DATASET_ZIP else 'Not uploaded (optional)')
if unknown: print('Ignored uploads:', unknown)

Saving clothing_model_bundle.zip to clothing_model_bundle.zip
Saving zodiac_wardrobe_colab_ready_app_FIXED.zip to zodiac_wardrobe_colab_ready_app_FIXED.zip
Application ZIP: zodiac_wardrobe_colab_ready_app_FIXED.zip
Model bundle ZIP: clothing_model_bundle.zip
Kaggle dataset ZIP: Not uploaded (optional)


## 3. Extract the ZIP files inside Colab and assemble the application

The app package and model bundle are extracted automatically. If you uploaded a Kaggle dataset ZIP, it is extracted to `/content/kaggle_dataset` too. The dataset is not needed for inference.

In [3]:
from pathlib import Path
from zipfile import ZipFile
import shutil

APP_EXTRACT = Path('/content/zodiac_app_extracted')
MODEL_EXTRACT = Path('/content/zodiac_model_extracted')
DATASET_EXTRACT = Path('/content/kaggle_dataset')
for folder in (APP_EXTRACT, MODEL_EXTRACT):
    if folder.exists(): shutil.rmtree(folder)
    folder.mkdir(parents=True, exist_ok=True)

def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    with ZipFile(zip_path) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError('Unsafe archive path: ' + member.filename)
        zf.extractall(destination)

safe_extract(APP_ZIP, APP_EXTRACT)
safe_extract(MODEL_ZIP, MODEL_EXTRACT)
if DATASET_ZIP is not None:
    if DATASET_EXTRACT.exists(): shutil.rmtree(DATASET_EXTRACT)
    DATASET_EXTRACT.mkdir(parents=True, exist_ok=True)
    print('Extracting optional Kaggle dataset. A large dataset may take several minutes...')
    safe_extract(DATASET_ZIP, DATASET_EXTRACT)
    print('styles.csv found:', bool(list(DATASET_EXTRACT.rglob('styles.csv'))))
    print('images folder found:', any(p.is_dir() and p.name.lower() == 'images' for p in DATASET_EXTRACT.rglob('*')))

backend_candidates = [p for p in APP_EXTRACT.rglob('main.py') if p.parent.name.lower() == 'backend']
if len(backend_candidates) != 1:
    raise FileNotFoundError(f'Expected one backend/main.py; found {len(backend_candidates)}.')
BACKEND_DIR = backend_candidates[0].parent
APP_DIR = BACKEND_DIR.parent
FRONTEND_PATH = APP_DIR / 'index.html'
if not FRONTEND_PATH.exists():
    raise FileNotFoundError('index.html not found next to backend folder.')

model_candidates = list(MODEL_EXTRACT.rglob('*.keras'))
label_candidates = [p for p in MODEL_EXTRACT.rglob('labels.json') if p.is_file()]
if len(model_candidates) != 1 or len(label_candidates) != 1:
    raise FileNotFoundError(f'Expected one .keras and one labels.json; found {len(model_candidates)} model(s), {len(label_candidates)} labels file(s).')

MODELS_DIR = BACKEND_DIR / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODELS_DIR / 'clothing_mobilenetv2.keras'
LABELS_PATH = MODELS_DIR / 'labels.json'
shutil.copy2(model_candidates[0], MODEL_PATH)
shutil.copy2(label_candidates[0], LABELS_PATH)

for cache_dir in APP_DIR.rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)
for pyc in APP_DIR.rglob('*.pyc'):
    try: pyc.unlink()
    except OSError: pass

print('Application root:', APP_DIR)
print('Model installed at:', MODEL_PATH)
print('Labels installed at:', LABELS_PATH)
print('All extraction and model placement completed inside Colab.')

Application root: /content/zodiac_app_extracted/zodiac_wardrobe_colab_ready_app
Model installed at: /content/zodiac_app_extracted/zodiac_wardrobe_colab_ready_app/backend/models/clothing_mobilenetv2.keras
Labels installed at: /content/zodiac_app_extracted/zodiac_wardrobe_colab_ready_app/backend/models/labels.json
All extraction and model placement completed inside Colab.


## 4. Configure frontend and FastAPI to use one ngrok URL

The frontend should call `/predict` on the same host rather than calling `localhost`, which would refer to the visitor's computer. FastAPI will serve the website at `/`.

In [4]:
html = FRONTEND_PATH.read_text(encoding='utf-8')
html = html.replace('http://127.0.0.1:8000/predict', '/predict')
html = html.replace('http://localhost:8000/predict', '/predict')
html = html.replace(
    'Clothing category is selected by you in this first prototype.',
    'MobileNetV2 predicts each clothing category. Review or edit the predicted category and detected colour when needed.'
)
html = html.replace(
    'This is a transparent rule-based prototype; the generative AI stylist and trained garment classifier are planned integration steps.',
    'Clothing categories are predicted by the trained MobileNetV2 CNN; outfit matching uses your uploaded wardrobe.'
)
html = html.replace('WORKING PROTOTYPE · V1', 'MOBILENETV2 · COLAB DEMO')
FRONTEND_PATH.write_text(html, encoding='utf-8')

main_path = BACKEND_DIR / 'main.py'
main_text = main_path.read_text(encoding='utf-8')
if 'from fastapi.responses import FileResponse' not in main_text:
    main_text = main_text.replace(
        'from fastapi.middleware.cors import CORSMiddleware',
        'from fastapi.middleware.cors import CORSMiddleware\nfrom fastapi.responses import FileResponse'
    )
if 'def serve_frontend()' not in main_text:
    route = "\n\n@app.get('/', include_in_schema=False)\ndef serve_frontend():\n    frontend_path = BASE_DIR.parent / 'index.html'\n    return FileResponse(frontend_path)\n"
    marker = '@app.get("/health")'
    if marker not in main_text:
        raise RuntimeError('Could not find /health route in backend/main.py')
    main_text = main_text.replace(marker, route + '\n\n' + marker)
main_path.write_text(main_text, encoding='utf-8')

if 'http://127.0.0.1:8000/predict' in FRONTEND_PATH.read_text(encoding='utf-8'):
    raise RuntimeError('A localhost API URL is still present in the frontend.')
print('Frontend and backend configured for same-origin hosting.')

Frontend and backend configured for same-origin hosting.


## 5. Load and validate the saved model

No training happens in this notebook. It loads the model you already trained. In the supplied training notebook, MobileNetV2 preprocessing is included inside the `.keras` model, so uploaded RGB images are passed as float32 values in the 0–255 range.

In [5]:
import json
import tensorflow as tf

with LABELS_PATH.open('r', encoding='utf-8') as f:
    labels = json.load(f)
if not isinstance(labels, dict) or not labels:
    raise ValueError('labels.json must be a non-empty dictionary mapping output indices to clothing names.')

loaded_model = tf.keras.models.load_model(MODEL_PATH)
output_shape = loaded_model.output_shape
if isinstance(output_shape, list):
    raise ValueError('This API supports a single-output classification model only.')
if output_shape[-1] != len(labels):
    raise ValueError(f'Model output classes ({output_shape[-1]}) do not match labels.json entries ({len(labels)}).')

print('Model loaded successfully.')
print('TensorFlow version:', tf.__version__)
print('Input shape:', loaded_model.input_shape)
print('Output shape:', loaded_model.output_shape)
print('Class mapping:')
for index, label in labels.items():
    print(f'  {index}: {label}')

Model loaded successfully.
TensorFlow version: 2.21.0
Input shape: (None, 224, 224, 3)
Output shape: (None, 7)
Class mapping:
  0: Dresses
  1: Jeans
  2: Shirts
  3: Shorts
  4: Track Pants
  5: Trousers
  6: Tshirts


## 6. Install the backend requirements

The application requirements file intentionally excludes TensorFlow because the previous setup cell already checked or installed a Python-compatible TensorFlow version. This prevents pip from downgrading TensorFlow to an incompatible release.

In [6]:
import subprocess, sys
requirements_path = BACKEND_DIR / 'requirements.txt'
if not requirements_path.exists():
    raise FileNotFoundError('backend/requirements.txt is missing.')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(requirements_path)])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok', 'requests'])
print('Backend dependencies installed.')

Backend dependencies installed.


## 7. Start FastAPI and verify the model

In [7]:
import subprocess, sys, time, requests
from pathlib import Path

try:
    if server_process.poll() is None:
        server_process.terminate()
        server_process.wait(timeout=8)
except NameError:
    pass
except Exception as exc:
    print('Previous server cleanup:', exc)

server_log_path = Path('/content/zodiac_fastapi.log')
server_log = open(server_log_path, 'w', encoding='utf-8')
server_process = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'],
    cwd=str(BACKEND_DIR), stdout=server_log, stderr=subprocess.STDOUT
)

health = None
for _ in range(60):
    try:
        response = requests.get('http://127.0.0.1:8000/health', timeout=4)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        time.sleep(1)
if health is None:
    server_log.flush()
    print(server_log_path.read_text(errors='ignore')[-8000:])
    raise RuntimeError('FastAPI did not start. Review the log above.')
print('Health response:', health)
if not health.get('model_loaded'):
    raise RuntimeError('API started but model_loaded is false: ' + str(health.get('message')))
print('SUCCESS: model_loaded = true')
print('Local preview: http://127.0.0.1:8000/')

Health response: {'status': 'ok', 'model_loaded': True, 'classes': ['Dresses', 'Jeans', 'Shirts', 'Shorts', 'Track Pants', 'Trousers', 'Tshirts'], 'message': 'Model is ready.'}
SUCCESS: model_loaded = true
Local preview: http://127.0.0.1:8000/


## 8. Start the temporary ngrok HTTPS tunnel

Create a free ngrok account and copy your authtoken from its dashboard. Enter it below; the input is hidden and the token is not printed.

In [8]:
import getpass
from pyngrok import ngrok

try:
    ngrok.kill()
except Exception:
    pass
ngrok_token = getpass.getpass('Enter your ngrok authtoken (hidden): ').strip()
if not ngrok_token:
    raise ValueError('An ngrok authtoken is required.')
ngrok.set_auth_token(ngrok_token)
del ngrok_token

tunnel = ngrok.connect(addr=8000, proto='http')
PUBLIC_URL = tunnel.public_url
print('Your temporary public website:', PUBLIC_URL)
print('API docs:', PUBLIC_URL + '/docs')

### MY Grok Token is: 3FPCTDU6sFyafY5NzabP2EgRKUA_29ktzuFbxukjYTd7nsWxe ###

Enter your ngrok authtoken (hidden): ··········
Your temporary public website: https://malformed-quilt-crook.ngrok-free.dev
API docs: https://malformed-quilt-crook.ngrok-free.dev/docs


## 9. Verify the public website and public model status

In [9]:
import json
health_response = requests.get(PUBLIC_URL + '/health', timeout=30)
health_response.raise_for_status()
print(json.dumps(health_response.json(), indent=2))
if not health_response.json().get('model_loaded'):
    raise RuntimeError('The public API reports that the model is not loaded.')
home_response = requests.get(PUBLIC_URL + '/', timeout=30)
home_response.raise_for_status()
print('Homepage HTTP status:', home_response.status_code)
print('Open your site:', PUBLIC_URL)

{
  "status": "ok",
  "model_loaded": true,
  "classes": [
    "Dresses",
    "Jeans",
    "Shirts",
    "Shorts",
    "Track Pants",
    "Trousers",
    "Tshirts"
  ],
  "message": "Model is ready."
}
Homepage HTTP status: 200
Open your site: https://malformed-quilt-crook.ngrok-free.dev


## 10. Optional: test one garment image

Upload one clear photo containing a single main garment. This calls the model API directly. It does not train the model.

In [ ]:
from google.colab import files
import mimetypes
sample_upload = files.upload()
if sample_upload:
    sample_name = next(iter(sample_upload))
    mime_type = mimetypes.guess_type(sample_name)[0] or 'image/jpeg'
    with open(sample_name, 'rb') as image_file:
        prediction = requests.post(
            'http://127.0.0.1:8000/predict',
            files={'image': (sample_name, image_file, mime_type)},
            timeout=90,
        )
    print('HTTP status:', prediction.status_code)
    print(prediction.json())
else:
    print('Skipped optional image test.')

Saving WhatsApp Image 2026-11-05 at 9.26.41 AM.jpeg to WhatsApp Image 2026-11-05 at 9.26.41 AM.jpeg
HTTP status: 200
{'predicted_label': 'Shirts', 'app_category': 'Top', 'confidence': 0.9988250136375427, 'confidence_percent': 99.88}


## 11. Stop the server and tunnel after your demo

Run this cell when you are finished. The public URL will stop working after the tunnel or Colab runtime stops.

In [ ]:
try:
    ngrok.disconnect(tunnel.public_url)
except Exception as exc:
    print('Tunnel cleanup:', exc)
try:
    ngrok.kill()
except Exception as exc:
    print('ngrok cleanup:', exc)
try:
    server_process.terminate()
    server_process.wait(timeout=10)
except Exception as exc:
    print('Server cleanup:', exc)
try:
    server_log.close()
except Exception:
    pass
print('Temporary server and tunnel stopped.')

Temporary server and tunnel stopped.


## Troubleshooting

- **Application ZIP not found:** Upload `zodiac_wardrobe_colab_ready_app.zip`. It must contain `backend/main.py` and `index.html`.
- **Model bundle ZIP not found:** Upload the `clothing_model_bundle.zip` generated by the training notebook. It must contain a `.keras` file and `labels.json`.
- **Kaggle dataset ZIP:** Optional for serving. If provided, the notebook extracts it to `/content/kaggle_dataset`; it is not used by prediction.
- **Model not loaded:** Ensure the model and labels came from the same training run.
- **Prediction errors:** Check `/health`, then test `/predict` through `/docs`.
- **Wrong predictions:** Use one clear garment per image. Catalogue training images and real phone photos can differ.
- **Colab disconnects:** The site and tunnel stop. Reconnect and rerun the notebook cells.
- **Privacy:** Anyone with the public ngrok URL can access the demo. Do not upload sensitive photographs or expose the token.